# Figure 6 CES and EXP distributions

The original boxplot sections for panels a and b are retained. Panels c and d are not included because their original plotting code was not recovered.

Input files are referenced by name; place the existing inputs in `INPUT_DIR` or point it to their location. Only figure files are saved if the plotting cells are run.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the existing input files for the selected panel.
INPUT_DIR = Path(".")
OUTPUT_DIR = Path("figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Panel a Positive CES distributions

Source: `CES_EXP_Aggregation_and_Ward_Clustering.ipynb`, cell 54. The existing CES percentages table replaces the original inline recomputation from annotated reviews.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.patches as mpatches # Import for custom legend handles

# Ensure font settings are loaded for Chinese characters
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS'] # Use the verified font name
plt.rcParams['axes.unicode_minus'] = False

ces_analysis_with_percentages = pd.read_csv(INPUT_DIR / "ces_category_sentiment_percentages_table.csv", index_col="scenic_spot")
positive_percent = [c for c in ces_analysis_with_percentages if c.endswith("_SNT_1_PCT")]
ces_analysis_with_percentages[positive_percent] = ces_analysis_with_percentages[positive_percent].round(2)


In [ ]:
# Filter for positive sentiment percentage columns
percentage_cols = [col for col in ces_analysis_with_percentages.columns if col.endswith('_SNT_1_PCT')]

# Create a DataFrame containing only the scenic spot and percentage columns
df_plot = ces_analysis_with_percentages[percentage_cols].reset_index()
df_plot = df_plot.rename(columns={'scenic_spot': 'Scenic Spot'})

# Melt the DataFrame to long format for seaborn
df_melted = df_plot.melt(id_vars=['Scenic Spot'], var_name='CES Category', value_name='Positive Percentage')

# Clean category names in df_melted to match display names
def clean_melted_category_name(name):
    clean_name = name.replace('_SNT_1_PCT', '')
    clean_name = clean_name.replace('_values', '')
    clean_name = clean_name.replace('_and_', ' & ') # Specific replacement for "_and_"
    clean_name = clean_name.replace('_', ' ')      # General replacement for remaining underscores
    return clean_name.title()

df_melted['CES Category'] = df_melted['CES Category'].apply(clean_melted_category_name)

# Define the order and display names for categories and collect actual data for boxplot
# These correspond to the 'cleaned_categories' from previous cells, ensuring order.
cleaned_categories_order = [
    'Aesthetic',
    'Cultural Heritage',
    'Inspiration',
    'Knowledge & Education',
    'Recreation & Visitation',
    'Sense Of Place',
    'Social Relations',
    'Spiritual & Religious Value'
]

ces_labels_display = [
    "Aesthetic",
    "Cultural\nHeritage",
    "Inspiration",
    "Knowledge &\nEducation",
    "Recreation &\nVisitation",
    "Sense of\nPlace",
    "Social\nRelations",
    "Spiritual &\nReligious"
]

data_for_boxplot = []
for clean_cat_name in cleaned_categories_order:
    # Ensure df_melted['CES Category'] matches these names
    data_for_boxplot.append(df_melted[df_melted['CES Category'] == clean_cat_name]['Positive Percentage'].values)

# Color palette from the reference code
colors = [
    "#4E79A7",  # Aesthetic
    "#F28E2B",  # Cultural Heritage
    "#59A14F",  # Inspiration
    "#E15759",  # Knowledge & Education
    "#B07AA1",  # Recreation & Visitation
    "#9C755F",  # Sense of Place
    "#EDC948",  # Social Relations
    "#76B7B2"   # Spiritual & Religious
]

# Set up the matplotlib figure - Reduced width from 12 to 8.5 to bring boxes closer together
fig, ax = plt.subplots(figsize=(8, 6.8))

# Create the box plot
boxplot = ax.boxplot(
    data_for_boxplot, # Use data extracted per category
    patch_artist=True,
    widths=0.5, # Adjusted box width for the narrower figure
    showfliers=False,   # Do not show default outliers
    medianprops=dict(
        color="#222222",
        linewidth=1.6
    ),
    whiskerprops=dict(
        color="#666666",
        linewidth=1.2
    ),
    capprops=dict(
        color="#666666",
        linewidth=1.2
    ),
    boxprops=dict(
        linewidth=1.2,
        color="#666666" # Default line color for box structure
    )
)

# Color each box and its outline by category, with adjusted alpha
for i, patch in enumerate(boxplot["boxes"]):
    color = colors[i % len(colors)]
    patch.set_facecolor(color)
    patch.set_alpha(0.3) # User feedback: alpha adjusted to 0.3
    patch.set_edgecolor(color) # Box outline colored by category

# Add jittered points (stripplot logic)
rng = np.random.default_rng(42)
for i, (values, color) in enumerate(zip(data_for_boxplot, colors), start=1):
    # Filter out NaN values for scatter plot to avoid errors
    valid_values = values[~np.isnan(values)]
    x_positions = rng.normal(
        loc=i,
        scale=0.055,
        size=len(valid_values)
    )
    ax.scatter(
        x_positions,
        valid_values,
        s=22, # Point size
        alpha=0.7, # User feedback: stripplot point transparency adjusted to 0.7
        color=color,
        edgecolor="white",
        linewidth=0.5,
        zorder=3
    )

# Add mean markers
means = [np.mean(values) for values in data_for_boxplot]
mean_marker_scatter = ax.scatter(
    range(1, len(data_for_boxplot) + 1),
    means,
    marker="D", # Diamond marker for mean
    s=42,
    color="#222222", # Dark grey for mean marker
    edgecolor="white",
    linewidth=0.7,
    zorder=4
)

# Axis labels and title
ax.set_title(
    "Distribution of Positive CES Category Percentages across Parks",
    fontsize=16,
    pad=14,
    weight="bold"
)

ax.set_ylabel(
    "Positive percentage (%)",
    fontsize=12
)

ax.set_xlabel(
    "CES category",
    fontsize=12,
    labelpad=10
)

ax.set_xticks(range(1, len(ces_labels_display) + 1))
ax.set_xticklabels(
    ces_labels_display, # Use display labels with newlines
    fontsize=10
)

ax.tick_params(
    axis="y",
    labelsize=10
)

ax.set_ylim(-3, 100) # Set y-axis limits as in reference

# Clean figure style
# Only horizontal grid lines
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    alpha=0.35
)

ax.grid(
    axis="x",
    visible=False
)

# Remove top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Subdue left and bottom spines
ax.spines["left"].set_alpha(0.4)
ax.spines["bottom"].set_alpha(0.4)

# Custom Legend
legend_handles = []
legend_labels = []

# Add legend entries for each category box
for i, cat_label in enumerate(ces_labels_display):
    legend_handles.append(mpatches.Patch(color=colors[i], alpha=0.3, ec=colors[i]))
    legend_labels.append(cat_label.replace('\n', ' ')) # Remove newlines for legend label

# Add mean marker to legend
legend_handles.append(mean_marker_scatter)
legend_labels.append("Mean")

ax.legend(
    legend_handles,
    legend_labels,
    frameon=False,
    loc="upper right",
    fontsize=10,
    bbox_to_anchor=(1.05, 1),
    ncol=1 # Display in one column since figure is narrower
)

plt.tight_layout() # Adjust layout to prevent labels from overlapping

# Save the plot
output_directory = str(OUTPUT_DIR)
os.makedirs(output_directory, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
boxplot_path = os.path.join(output_directory, f'ces_category_positive_percentage_boxplot_polished_{timestamp}.png') # Updated filename
plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
plt.close() # Close the figure to free memory

print(f"箱型图已保存至: {boxplot_path}")


## Panel b Positive and negative EXP distributions

Source: `CES_EXP_Aggregation_and_Ward_Clustering.ipynb`, cell 75, with the evidence-filtered percentages from the June 26, 2026 export in cell 69.


In [ ]:
comfort_categories = ["Accessibility", "Safety", "Usability", "Comfort"]
comfort_analysis_with_percentages = pd.read_csv(
    INPUT_DIR / "comfort_sentiment_percentages_table_20260626155035.csv", index_col="scenic_spot"
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.patches as mpatches

# Ensure font settings are loaded for Chinese characters
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('开始生成正负向对比的舒适度箱型图...')

# --- Prepare Data ---
# Positive Data
percentage_cols_comfort_pos = [f'{cat}_SNT_1_PCT' for cat in comfort_categories]
df_pos = comfort_analysis_with_percentages[percentage_cols_comfort_pos].reset_index()
df_pos = df_pos.rename(columns={'scenic_spot': 'Scenic Spot'})
df_pos_melted = df_pos.melt(id_vars=['Scenic Spot'], var_name='Comfort Category', value_name='Percentage')
df_pos_melted['Sentiment'] = 'Positive'

# Clean names for positive
def clean_comfort_category_name_pos(name):
    return name.replace('_SNT_1_PCT', '').title()
df_pos_melted['Comfort Category'] = df_pos_melted['Comfort Category'].apply(clean_comfort_category_name_pos)

# Negative Data
percentage_cols_comfort_neg = [f'{cat}_SNT_0_PCT' for cat in comfort_categories]
df_neg = comfort_analysis_with_percentages[percentage_cols_comfort_neg].reset_index()
df_neg = df_neg.rename(columns={'scenic_spot': 'Scenic Spot'})
df_neg_melted = df_neg.melt(id_vars=['Scenic Spot'], var_name='Comfort Category', value_name='Percentage')
df_neg_melted['Sentiment'] = 'Negative'

# Clean names for negative
def clean_comfort_category_name_neg(name):
    return name.replace('_SNT_0_PCT', '').title()
df_neg_melted['Comfort Category'] = df_neg_melted['Comfort Category'].apply(clean_comfort_category_name_neg)

# Combine Data
df_combined = pd.concat([df_pos_melted, df_neg_melted], ignore_index=True)

# Define the order for x-axis
comfort_labels_display = [
    'Accessibility',
    'Safety',
    'Usability',
    'Comfort'
]

# Colors from original plots
colors_comfort_pos = sns.color_palette('viridis', n_colors=len(comfort_categories))
colors_comfort_neg = sns.color_palette('Reds', n_colors=len(comfort_categories) + 2)[2:]

# Set up the matplotlib figure
fig_combined, ax_combined = plt.subplots(figsize=(10, 7))

# Use seaborn boxplot for easier grouping
# We will customize colors manually to match user request
ax_combined = sns.boxplot(
    data=df_combined,
    x='Comfort Category',
    y='Percentage',
    hue='Sentiment',
    order=comfort_labels_display,
    showfliers=False,
    width=0.4,
    boxprops=dict(linewidth=1.2),
    medianprops=dict(color="#222222", linewidth=1.6),
    whiskerprops=dict(color="#666666", linewidth=1.2),
    capprops=dict(color="#666666", linewidth=1.2),
    ax=ax_combined
)

# Customize colors for each box to match the original style
# Seaborn creates boxes in order of hue (Positive, Negative) per category.
# The patches are flat list: [Pos_Cat1, Pos_Cat2..., Neg_Cat1, Neg_Cat2...]
num_cats = len(comfort_labels_display)
boxes = ax_combined.patches

for i in range(num_cats):
    # Positive boxes
    pos_box = boxes[i]
    pos_color = colors_comfort_pos[i]
    pos_box.set_facecolor(pos_color)
    pos_box.set_alpha(0.3)
    pos_box.set_edgecolor(pos_color)

    # Negative boxes
    neg_box = boxes[i + num_cats]
    neg_color = colors_comfort_neg[i]
    neg_box.set_facecolor(neg_color)
    neg_box.set_alpha(0.5)
    neg_box.set_edgecolor(neg_color)

# Add jittered points manually for exact color matching and placement
rng = np.random.default_rng(42)
for cat_idx, cat in enumerate(comfort_labels_display):
    # Positive points
    pos_values = df_combined[(df_combined['Comfort Category'] == cat) & (df_combined['Sentiment'] == 'Positive')]['Percentage'].dropna().values
    if len(pos_values) > 0:
        # Adjusted loc offset from 0.175 to 0.1 to match width=0.4
        x_pos_jitter = rng.normal(loc=cat_idx - 0.1, scale=0.04, size=len(pos_values))
        ax_combined.scatter(x_pos_jitter, pos_values, s=22, alpha=0.7, color=colors_comfort_pos[cat_idx], edgecolor="white", linewidth=0.5, zorder=3)

    # Negative points
    neg_values = df_combined[(df_combined['Comfort Category'] == cat) & (df_combined['Sentiment'] == 'Negative')]['Percentage'].dropna().values
    if len(neg_values) > 0:
        # Adjusted loc offset from 0.175 to 0.1 to match width=0.4
        x_neg_jitter = rng.normal(loc=cat_idx + 0.1, scale=0.04, size=len(neg_values))
        ax_combined.scatter(x_neg_jitter, neg_values, s=22, alpha=0.7, color=colors_comfort_neg[cat_idx], edgecolor="white", linewidth=0.5, zorder=3)

# Calculate and plot means
# Get the positions of the boxes to place the mean markers correctly
# Calculate offsets based on dodge
hue_order = ['Positive', 'Negative']
for cat_idx, cat in enumerate(comfort_labels_display):
    for hue_idx, sentiment in enumerate(hue_order):
        # Get data
        values = df_combined[(df_combined['Comfort Category'] == cat) & (df_combined['Sentiment'] == sentiment)]['Percentage'].dropna()
        mean_val = np.mean(values)

        # Determine x position
        # Boxplot width is 0.4. Two boxes per group -> each takes 0.2 roughly.
        # Centers are offset from integer positions.
        # Adjusted offset from 0.175 to 0.1
        offset = -0.1 if sentiment == 'Positive' else 0.1
        x_pos = cat_idx + offset

        ax_combined.scatter(
            x_pos,
            mean_val,
            marker="D",
            s=42,
            color="#222222",
            edgecolor="white",
            linewidth=0.7,
            zorder=4
        )

# Axis labels and title
ax_combined.set_title(
    "Distribution of Positive vs Negative Comfort Category Percentages",
    fontsize=16,
    pad=14,
    weight="bold"
)
ax_combined.set_ylabel("Percentage (%)", fontsize=12)
ax_combined.set_xlabel("Comfort Category", fontsize=12, labelpad=10)
ax_combined.tick_params(axis="y", labelsize=10)
ax_combined.tick_params(axis="x", labelsize=11)

# Clean figure style
ax_combined.grid(axis="y", linestyle="--", linewidth=0.8, alpha=0.35)
ax_combined.grid(axis="x", visible=False)
ax_combined.spines["top"].set_visible(False)
ax_combined.spines["right"].set_visible(False)
ax_combined.spines["left"].set_alpha(0.4)
ax_combined.spines["bottom"].set_alpha(0.4)

# Custom Legend
legend_elements = [
    mpatches.Patch(color="gray", alpha=0.3, label='Positive Distribution'),
    mpatches.Patch(color="gray", alpha=0.5, label='Negative Distribution'),
    plt.scatter([], [], marker="D", s=42, color="#222222", edgecolor="white", label="Mean")
]
ax_combined.legend(handles=legend_elements, loc="upper right", frameon=False, fontsize=10)

plt.tight_layout()

# Save the plot
output_directory_comfort = str(OUTPUT_DIR)
os.makedirs(output_directory_comfort, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
combined_boxplot_path = os.path.join(output_directory_comfort, f'comfort_category_combined_percentage_boxplot_{timestamp}.png')
fig_combined.savefig(combined_boxplot_path, dpi=300, bbox_inches='tight')
plt.close(fig_combined)

print(f"正负向对比舒适度箱型图已生成并保存至: {combined_boxplot_path}")
